# Cart–pendulum LQR (explained step by step)

**What you are building:** a computer program that pushes the cart left/right so the pendulum stays balanced upright (like an inverted broomstick on your hand).

**Big picture (no heavy math):**

1. We build a *simulation* of a cart on a rail with a pendulum.
2. We pick the pose we want to hold: pendulum straight up, cart at the middle.
3. We tell the simulator: “If I’m slightly off that pose, how does the world respond to a small push on the cart?” That gives two tables of numbers called **A** and **B**.
4. We choose what we care about (stay upright! don’t shove the cart too hard!) using **Q** and **R**.
5. A standard formula computes **K** — the recipe that turns “how wrong am I?” into “how hard should I push the cart?”
6. Each frame of the animation: measure error → compute push → step physics → repeat.

Run cells **top to bottom** in order.

## 1. Imports and setup

- **MuJoCo** = physics engine (simulated gravity, joints, forces).
- **NumPy** = working with lists of numbers (positions, forces).
- **SciPy** = one function to compute the control recipe **K**.
- **mediapy** = show the simulation as a video in the notebook.

In [ ]:
# If imports fail, install once in a terminal:
#   pip install mujoco mediapy matplotlib scipy

import os
import sys

import numpy as np          # arrays of numbers (positions, forces, matrices)
import scipy.linalg         # solves for the control gain K
import mujoco               # physics simulation

# On Windows, pick a graphics backend so MuJoCo can render (if you use the viewer).
if sys.platform == "win32" and "MUJOCO_GL" not in os.environ:
    os.environ.setdefault("MUJOCO_GL", "glfw")

import matplotlib.pyplot as plt

try:
    import mediapy as media   # display video in the notebook
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mediapy"])
    import mediapy as media

import shutil
from IPython.display import HTML, display
from matplotlib import animation as mpl_animation

# mediapy normally needs ffmpeg. If ffmpeg is missing, show video via matplotlib instead.
if not shutil.which("ffmpeg"):
    def _show_video_matplotlib(frames, fps=30, **kwargs):
        fig, ax = plt.subplots()
        ax.axis("off")
        im = ax.imshow(frames[0])
        interval_ms = 1000 / fps if fps else 33

        def _update(i):
            im.set_data(frames[i])
            return [im]

        ani = mpl_animation.FuncAnimation(
            fig, _update, frames=len(frames), interval=interval_ms, blit=True
        )
        plt.close(fig)
        display(HTML(ani.to_jshtml()))

    media.show_video = _show_video_matplotlib

# Print arrays in a readable way (3 decimal places).
np.set_printoptions(precision=3, suppress=True, linewidth=100)

## 2. Describe the robot in XML

MuJoCo models are written in XML. Think of it as a blueprint:

| Piece | Meaning |
|-------|--------|
| `cart_slide` | Cart moves horizontally on a rail |
| `pendulum_hinge` | Pendulum rotates on the cart |
| `motor cart_force` | We can apply a **horizontal force** on the cart (Newtons). That is our only control. |
| `ctrlrange -20 20` | Force is clamped between −20 N and +20 N |
| `upright_0deg` keyframe | Saved pose: cart at 0, pendulum angle = π rad (straight up) |
| `upright_2deg` keyframe | Same but pendulum tipped 2° (used later as a test) |

Angles are in **radians**. π radians ≈ 180° = upright for this model.

In [ ]:
xml = """<mujoco model="cart_pendulum">
  <compiler angle="radian" autolimits="true"/>
  <!-- Physics step ~240 Hz. "implicitfast" integrator is required later for mjd_transitionFD. -->
  <option timestep="0.004166666666666667" gravity="0 0 -9.80665" integrator="implicitfast"/>

  <default>
    <joint damping="0.04" armature="0.001"/>
    <geom rgba="0.55 0.55 0.6 1" friction="0.2 0.05 0.001"/>
  </default>

  <worldbody>
    <light name="main_light" pos="0 -1 1" dir="0 1 -1" diffuse="1 1 1"/>
    <camera name="side" pos="0 -1.2 0.35" xyaxes="1 0 0 0 0 1"/>
    <geom name="rail" type="box" size="1 0.02 0.01" pos="0 0 0"
          rgba="0.35 0.35 0.4 1" contype="0" conaffinity="0"/>

    <body name="cart" pos="0 0 0.05">
      <joint name="cart_slide" type="slide" axis="1 0 0" damping="0.5" limited="false"/>
      <geom name="cart_geom" type="box" size="0.06 0.04 0.03" mass="0.8"/>
      <body name="pendulum" pos="0 0 0">
        <joint name="pendulum_hinge" type="hinge" axis="0 1 0" pos="0 0 0"/>
        <geom name="rod" type="capsule" fromto="0 0 0 0 0 -0.35" size="0.004" mass="0.12"/>
        <geom name="bob" type="sphere" size="0.03" pos="0 0 -0.35" mass="0.12"/>
      </body>
    </body>
  </worldbody>

  <actuator>
    <motor name="cart_force" joint="cart_slide" gear="1"
           ctrllimited="true" ctrlrange="-50 50"/>
  </actuator>

  <keyframe>
    <key name="upright_0deg" qpos="0 3.14159265359" ctrl="0"/>
    <key name="upright_2deg" qpos="0 3.17649923863" ctrl="0"/>
    <key name="upright_minus_45deg" qpos="0 2.356194490" ctrl="0"/>
  </keyframe>
</mujoco>
"""

## 3. Load the model and remember the “goal pose”

- **`model`** = fixed description (masses, joint limits, actuator layout).
- **`data`** = changing state while simulating (positions, velocities, forces).
- **`qpos0`** = goal joint positions: `[cart_x, pendulum_angle]` at upright.
- **`ctrl0`** = baseline push on the cart at the goal. Here it is **zero** (no force needed when perfectly balanced and still).

All later control is: *baseline push* plus *correction based on how far we are from the goal*.

In [ ]:
# Build model and simulation state from the XML string.
model = mujoco.MjModel.from_xml_string(xml)
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model)  # used at the end to make video frames

# --- Goal pose: perfectly upright (upright_0deg keyframe) ---
key_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_KEY, "upright_0deg")
mujoco.mj_resetDataKeyframe(model, data, key_id)  # load saved qpos into data

qpos0 = data.qpos.copy()   # [cart position (m), pendulum angle (rad)]
ctrl0 = np.zeros(model.nu)  # nu = number of actuators (here: 1 motor → one number)

print("Goal cart position (m):", qpos0[0])
print("Goal pendulum angle (rad):", qpos0[1], "  (~pi = upright)")
print("Baseline motor force ctrl0 (N):", ctrl0)

# Camera for the final video (orbit / zoom settings).
camera = mujoco.MjvCamera()
mujoco.mjv_defaultFreeCamera(model, camera)
camera.distance = 2

## 4. What should the controller care about? (Q and R)

LQR needs two weight tables. Intuition only:

**State error vector `dx`** (4 numbers each timestep):

1. How far is the cart from its goal position?
2. How far is the pendulum angle from upright?
3. How fast is the cart moving?
4. How fast is the pendulum spinning?

**Q** — “How bad is each kind of error?” Bigger number = try harder to fix that error.

We use `[1, 100, 1, 10]` on those four errors: **pendulum angle matters most** (100), then pendulum speed (10), cart position and speed matter less (1).

**R** — “How bad is pushing hard on the cart?” Bigger R = gentler, softer pushes. We use `0.1` on the single motor.

You can tune these: if the pendulum falls, increase the pendulum entries in Q. If the cart slams around, increase R.

In [ ]:
nu = model.nu   # number of controls (motors) = 1
nv = model.nv   # number of velocities = 2 (cart + pendulum)

# Cost of using force on the motor (penalize hard pushes).
R = 0.01 * np.eye(nu)

# Cost of being away from the goal state [cart_pos, pend_angle, cart_vel, pend_vel].
Q = np.diag([
    500.0,    # cart position error
    100.0,  # pendulum angle error  ← most important
    1.0,    # cart velocity
    10.0,   # pendulum angular velocity
])

print("Q (penalty on each error):", np.diag(Q))
print("R (penalty on force):", R)

## 5. Approximate “physics near the goal” (matrices A and B)

The real pendulum physics is nonlinear (sin/cos, etc.). **LQR only uses a local linear approximation** around the upright pose:

> “If I’m almost at the goal and apply a small extra force, what happens on the next tiny timestep?”

MuJoCo estimates that with **`mjd_transitionFD`** by nudging each variable slightly and measuring the change. It fills:

- **A** — how the 4 error numbers evolve from the current errors (without extra push)
- **B** — how the 4 error numbers change when we add a tiny bit of cart force

You do not need to derive these by hand; the function does numerical experiments inside the simulator.

In [ ]:
# Put the simulator exactly at the linearization point (goal pose, not moving, baseline force).
mujoco.mj_resetData(model, data)
data.qpos = qpos0
data.qvel[:] = 0
data.ctrl = ctrl0

# Allocate empty tables; mjd_transitionFD will fill them.
# Size: state has 2 positions + 2 velocities = 4 numbers → 4×4 matrix A.
#       1 motor → 4×1 matrix B.
A = np.zeros((2 * nv, 2 * nv))
B = np.zeros((2 * nv, nu))

epsilon = 1e-6       # size of tiny nudge for finite differences
flg_centered = True  # more accurate symmetric difference
mujoco.mjd_transitionFD(model, data, epsilon, flg_centered, A, B, None, None)

print("A shape (state × state):", A.shape)
print("B shape (state × motors):", B.shape)

## 6. Compute the control recipe **K**

Given A, B, Q, R, a standard optimal-control solver finds **K** so that:

```text
extra_force = -K @ dx
```

Read `@` as “combine the four errors using these weights.” Negative sign means: if the pendulum leans right, push the cart the way that catches it.

Under the hood: `solve_discrete_are` solves the **algebraic Riccati equation** (a built-in formula from control theory). You can treat it as a black box that returns the best linear controller for our Q and R.

**K** has shape `(1, 4)` here: one motor, four error signals.

In [ ]:
# P is an internal intermediate matrix from the Riccati solver (not used again below).
P = scipy.linalg.solve_discrete_are(A, B, Q, R)

# Feedback gain: how strongly each error drives the motor.
K = np.linalg.inv(R + B.T @ P @ B) @ B.T @ P @ A

print("K (motor force per unit error):", K)
print("  K[0,0] × cart position error  → force contribution")
print("  K[0,1] × pendulum angle error → force contribution (usually largest)")
print("  K[0,2] × cart velocity error  → force contribution")
print("  K[0,3] × pendulum rate error  → force contribution")

## 7. Run the closed-loop simulation (the actual balancer)

**Start pose:** `upright_2deg` — pendulum only 2° off upright (small wobble).

**Each physics step:**

1. Measure how far we are from the goal (`dx`).
2. Compute motor force: `ctrl = ctrl0 - K @ dx` (then clamp to ±20 N).
3. Step MuJoCo one timestep forward.
4. Occasionally save a picture for the video.

If **K** and Q/R are reasonable, the pendulum should stay near upright and the cart will move under the rod to catch falls.

In [ ]:
DURATION = 2          # seconds of simulated time
FRAMERATE = 60        # video frames per second

# --- Initial condition: slightly tipped (2°), cart at center, not moving ---
key_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_KEY, "upright_minus_45deg")
mujoco.mj_resetDataKeyframe(model, data, key_id)
data.qvel[:] = 0

dq = np.zeros(nv)  # buffer for position difference (2 joints)
ctrlrange = model.actuator_ctrlrange[0]  # [min_force, max_force] for the motor

frames = []
while data.time < DURATION:
    # --- 1. How far are we from the goal? ---
    # mj_differentiatePos: safe angle difference for the hinge (handles wrap-around).
    mujoco.mj_differentiatePos(model, dq, 1, qpos0, data.qpos)
    dx = np.hstack((dq, data.qvel))  # 4 numbers: 2 position errors + 2 velocities

    # --- 2. Control law: baseline force minus correction ---
    # K @ dx is a (1,) array; convert to a plain scalar before assigning into data.ctrl[0].
    force = float((ctrl0 - K @ dx).squeeze())
    data.ctrl[0] = float(np.clip(force, ctrlrange[0], ctrlrange[1]))

    # --- 3. Advance physics one timestep ---
    mujoco.mj_step(model, data)

    # --- 4. Save frames for video (~60 per second) ---
    if len(frames) < data.time * FRAMERATE:
        renderer.update_scene(data, camera)
        frames.append(renderer.render())

# How far from upright (in degrees) at the end?
pend_adr = model.joint("pendulum_hinge").qposadr[0]
tilt_deg = np.degrees(data.qpos[pend_adr] - np.pi)
print(f"Final tilt from upright: {tilt_deg:.2f} degrees  (near 0 = success)")
print(f"Final cart position (m): {data.qpos[0]:.4f}")

media.show_video(frames, fps=FRAMERATE)

## 8. Glossary (quick reference)

| Name | Plain English |
|------|----------------|
| `qpos` | Joint positions (where things are) |
| `qvel` | Joint speeds |
| `ctrl` | Command to the motor (here: horizontal force in N) |
| `qpos0`, `ctrl0` | Goal pose and baseline command we linearize around |
| `dx` | “Error vector” — how far and how fast we are from the goal |
| `Q` | Penalties on errors (what we want small) |
| `R` | Penalty on pushing hard |
| `A`, `B` | Local linear model of physics near the goal |
| `K` | Gains: converts errors into extra motor force |
| LQR | Linear Quadratic Regulator — optimal linear controller for this setup |

**Real hardware note:** your physical rig may use a *position* slider, not a direct force motor. This notebook uses a force motor because it matches the MuJoCo LQR tutorial; deploying on the real cart needs an extra layer (turn LQR force into position/velocity commands).